# R8 Colab Scanner
Google Drive上のコーパスを一括スキャンしてCSV出力する。

## セットアップ（初回のみ）
1. Google Driveに `R8/` フォルダを作成
2. `R8/` に `r8.py` をアップロード
3. `R8/corpus/` にスキャン対象の `.txt` ファイルを配置

```
Google Drive/
  R8/
    r8.py
    corpus/
      book/
        kimiwa_naze_hataraku/
          ch1.txt
          ch2.txt
        kyouiku_houkai/
          ch1.txt
      web/
        WEB_064.txt
        SN_086.txt
    results/
      (CSVが自動出力される)
```

## Step 1: Driveマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: R8をインポート

In [ ]:
import sys
import os

R8_DIR = '/content/drive/MyDrive/R8'

# r8.pyの存在確認
assert os.path.exists(os.path.join(R8_DIR, 'r8.py')), f'r8.py が {R8_DIR} に見つかりません'

# r8.pyをimportできるようにパスを追加
if R8_DIR not in sys.path:
    sys.path.insert(0, R8_DIR)

import r8
print(f'R8 Analyzer {r8.VERSION} loaded')

## Step 3: スキャン対象フォルダを指定して一括実行
`SCAN_DIR` を変更すれば任意のフォルダをスキャンできる。

In [ ]:
import csv
from datetime import datetime

# === 設定 ===
# スキャン対象フォルダ（変更可）
SCAN_DIR = os.path.join(R8_DIR, 'corpus')
# 結果出力先
RESULTS_DIR = os.path.join(R8_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# === txtファイル収集 ===
txt_files = []
for root, dirs, files in os.walk(SCAN_DIR):
    for f in sorted(files):
        if f.endswith('.txt') and f != 'targets.txt':
            txt_files.append(os.path.join(root, f))

print(f'スキャン対象: {len(txt_files)} ファイル')
for f in txt_files:
    print(f'  {os.path.relpath(f, R8_DIR)}')

In [ ]:
# === スキャン実行 & CSV出力 ===
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
csv_path = os.path.join(RESULTS_DIR, f'scan_{timestamp}.csv')

categories = list(r8.WEIGHTS.keys())
header = ['file', 'cmi', 'level'] + categories

results = []
for filepath in txt_files:
    name = os.path.relpath(filepath, R8_DIR)
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        if len(text.strip()) < 10:
            print(f'  SKIP (短すぎ): {name}')
            continue
        raw = r8.analyze(text)
        ri = {cat: min(raw.get(cat, 0) / r8.THRESHOLDS[cat], 1.0) for cat in r8.WEIGHTS}
        cmi = round(sum(r8.WEIGHTS[c] * ri[c] * 100 for c in r8.WEIGHTS), 1)
        level = r8.cmi_level(cmi).strip()
        row = [name, cmi, level] + [round(ri[c], 3) for c in categories]
        results.append(row)
        print(f'  {level:12s} CMI {cmi:5.1f}  {name}')
    except Exception as e:
        print(f'  ERROR: {name} -> {e}')

# CSV書き出し
with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(results)

print(f'\n完了: {len(results)} ファイル → {csv_path}')

## Step 4 (任意): 単体スキャン
1ファイルだけ詳細レポートを見たいとき。

In [ ]:
# ファイルパスを指定（変更可）
target_file = os.path.join(R8_DIR, 'corpus/book/kimiwa_naze_hataraku/ch1.txt')

with open(target_file, 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()
r8.report(text, os.path.basename(target_file))